**Навигация по уроку**

1. [Библиотеки визуализации данных в Python](https://colab.research.google.com/drive/1IbLhzlqiJhhPAErmdQ9wuIUdDaSRlNUF)
2. [Задача об акциях Tesla](https://colab.research.google.com/drive/1jukOs54u301WtyQS-SbRxWxDstTCjEBq)
3. Домашняя работа

Используя датасет о стоимости акций Сбербанка с 01.01.2013 года:

https://storage.yandexcloud.net/academy.ai/SBER.csv

визуализируйте индикатор "Полосы Боллинджера", проанализируйте график, и предложите вариант торговли акциями Сбербанка с помощью этого инструмента.

**Подсказка.**
Индикатор выглядит как полоса из трех линий:

* линия посередине — это простая скользящая средняя (SMA) с периодом `ma_size`, обычно около 20 дней;

* верхняя и нижняя линии (BB) — построены на основе SMА, но к нему добавлено стреднеквадратичное отклонение:

```
 SMA = data['close'].rolling(ma_size).mean()
 BB_UP = SMA + data['close'].rolling(ma_size).std() * bol_size
 BB_DOWN = SMA - data['close'].rolling(ma_size).std() * bol_size
   
```

где bol_size - ширина коридора, подбирается по графику. Выберите такое его значение, чтобы по графику можно было принимать торговые решения.

In [ ]:
# Загрузим все необходимые библиотеки
import os
import numpy as np
import pandas as pd

import plotly as py
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

In [ ]:
!wget https://storage.yandexcloud.net/academy.ai/SBER.csv

--2024-11-09 01:58:03--  https://storage.yandexcloud.net/academy.ai/SBER.csv
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 180264 (176K) [text/csv]
Saving to: ‘SBER.csv’

SBER.csv            100%[===================>] 176.04K   419KB/s    in 0.4s    

2024-11-09 01:58:04 (419 KB/s) - ‘SBER.csv’ saved [180264/180264]



In [ ]:
df = pd.read_csv("./SBER.csv") # wget скачал в рабочую директорию файл с датасетом и мы его уже можем использовать
df.head()

,DATE;OPEN;HIGH;LOW;CLOSE;VOL
0,20130108;96.5000000;98.5000000;96.1200000;98.3...
1,20130109;98.4100000;98.6500000;97.8100000;98.2...
2,20130110;98.3500000;98.5500000;97.9600000;98.4...
3,20130111;98.8000000;99.7600000;98.4800000;99.5...
4,20130114;99.7200000;101.2700000;99.5700000;100...


In [ ]:
df[['DATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL']] = df['DATE;OPEN;HIGH;LOW;CLOSE;VOL'].str.split(';', expand=True)

del df['DATE;OPEN;HIGH;LOW;CLOSE;VOL']

df.head()


,DATE,OPEN,HIGH,LOW,CLOSE,VOL
0,20130108,96.5000000,98.5000000,96.1200000,98.3700000,92329970
1,20130109,98.4100000,98.6500000,97.8100000,98.2300000,59776760
2,20130110,98.3500000,98.5500000,97.9600000,98.4100000,47627940
3,20130111,98.8000000,99.7600000,98.4800000,99.5600000,71837780
4,20130114,99.7200000,101.2700000,99.5700000,100.9900000,83853170


In [ ]:
df['DATE'] = pd.to_datetime(df['DATE'])
df.index = range(len(df))

In [ ]:
df.head()


,DATE,OPEN,HIGH,LOW,CLOSE,VOL
0,2013-01-08,96.5000000,98.5000000,96.1200000,98.3700000,92329970
1,2013-01-09,98.4100000,98.6500000,97.8100000,98.2300000,59776760
2,2013-01-10,98.3500000,98.5500000,97.9600000,98.4100000,47627940
3,2013-01-11,98.8000000,99.7600000,98.4800000,99.5600000,71837780
4,2013-01-14,99.7200000,101.2700000,99.5700000,100.9900000,83853170


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2718 entries, 0 to 2717
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   DATE    2718 non-null   datetime64[ns]
 1   OPEN    2718 non-null   float64       
 2   HIGH    2718 non-null   float64       
 3   LOW     2718 non-null   float64       
 4   CLOSE   2718 non-null   float64       
 5   VOL     2718 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 127.5 KB


In [ ]:
df["OPEN"] = df['OPEN'].astype('float64')
df["HIGH"] = df['HIGH'].astype('float64')
df["LOW"] = df['LOW'].astype('float64')
df["CLOSE"] = df['CLOSE'].astype('float64')
df["VOL"] = df['VOL'].astype('float64')

In [ ]:
# Загрузим все необходимые библиотеки
import os
import numpy as np
import pandas as pd

import plotly as py
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

# Параметры
ma_size = 20
bol_size = 2

# Рассчитаем SMA, верхнюю и нижнюю границы
df['SMA'] = df['CLOSE'].rolling(window=ma_size).mean()
df['BB_UP'] = df['SMA'] + (df['HIGH'].rolling(window=ma_size).std() * bol_size)
df['BB_DOWN'] = df['SMA'] - (df['CLOSE'].rolling(window=ma_size).std() * bol_size)

# Создаем объект Figure
fig = go.Figure()

# Добавляем свечной график
fig.add_trace(go.Candlestick(x=df['DATE'],
                              open=df['OPEN'],
                              high=df['HIGH'],
                              low=df['LOW'],
                              close=df['CLOSE'],
                              name='Свечной график'))

# Добавляем скользящую среднюю
fig.add_trace(go.Scatter(x=df['DATE'], y=df['SMA'], mode='lines', name='SMA', line=dict(color='blue')))

# Добавляем верхнюю и нижнюю границы Боллинджера
fig.add_trace(go.Scatter(x=df['DATE'], y=df['BB_UP'], mode='lines', name='BB_UP', line=dict(color='red', dash='dash')))
fig.add_trace(go.Scatter(x=df['DATE'], y=df['BB_DOWN'], mode='lines', name='BB_DOWN', line=dict(color='green', dash='dash')))

# Настраиваем отображение графика
fig.update_layout(title='Свечной график с SMA и полосами Боллинджера',
                  xaxis_title='Дата',
                  yaxis_title='Цена',
                  xaxis_rangeslider_visible=False)

# Показываем график
fig.show()


Анализ графика:
Здесь применяется базовое правило: осуществляйте приобретение на уровне нижней границы, а реализацию — на уровне средней или верхней границы. Чтобы избежать преждевременного закрытия позиции, необходимо перемещать стоп-ордер ниже вновь установленного минимума или выше нового максимума в случае осуществления короткой сделки.